In [1]:
!pip install yfinance sqlalchemy

import yfinance as yf
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from google.colab import files

**Create Folders**

In [2]:
os.makedirs("data/processed", exist_ok=True)

**Define Tickers**

In [3]:
tickers = ["AAPL","MSFT","NVDA","JPM","XOM",
    "GLD","TLT","SPY",
    "AMZN","GOOGL","META","TSLA"]

data = yf.download(
    tickers,
    start="2015-01-01",
    auto_adjust=True,
    progress=False
)

prices = data["Close"].reset_index()

In [4]:
prices.head()

Ticker,Date,AAPL,AMZN,GLD,GOOGL,JPM,META,MSFT,NVDA,SPY,TLT,TSLA,XOM
0,2015-01-02,24.214891,15.4260,114.080002,26.260458,46.274315,77.839149,39.767700,0.482985,170.125031,93.712059,14.620667,57.533417
1,2015-01-05,23.532717,15.1095,115.800003,25.760096,44.837734,76.588966,39.401997,0.474828,167.052612,95.184158,14.006000,55.959217
2,2015-01-06,23.534943,14.7645,117.120003,25.124350,43.675140,75.557076,38.823685,0.460432,165.479111,96.899055,14.085333,55.661709
3,2015-01-07,23.864950,14.9210,116.430000,25.050457,43.741776,75.557076,39.316944,0.459232,167.541199,96.707741,14.063333,56.225704
4,2015-01-08,24.781897,15.0230,115.940002,25.137735,44.719242,77.571266,40.473564,0.476507,170.514221,95.427055,14.041333,57.161556


**Create Price Table**

In [5]:
fact_asset_prices = prices.melt(
    id_vars="Date",
    var_name="ticker",
    value_name="close_price"
)

fact_asset_prices = fact_asset_prices.rename(columns={
    "Date": "price_date"
})

fact_asset_prices = fact_asset_prices.dropna()

fact_asset_prices.head()

,price_date,ticker,close_price
0,2015-01-02,AAPL,24.214891
1,2015-01-05,AAPL,23.532717
2,2015-01-06,AAPL,23.534943
3,2015-01-07,AAPL,23.864950
4,2015-01-08,AAPL,24.781897


**Create Returns Table**

In [6]:
fact_asset_returns = fact_asset_prices.copy()

fact_asset_returns = fact_asset_returns.sort_values(
    ["ticker", "price_date"]
)

fact_asset_returns["daily_return"] = (
    fact_asset_returns
    .groupby("ticker")["close_price"]
    .pct_change()
)

fact_asset_returns = fact_asset_returns.dropna()

fact_asset_returns = fact_asset_returns.rename(columns={
    "price_date": "return_date"
})

fact_asset_returns.head()

,return_date,ticker,close_price,daily_return
1,2015-01-05,AAPL,23.532717,-0.028172
2,2015-01-06,AAPL,23.534943,0.000095
3,2015-01-07,AAPL,23.864950,0.014022
4,2015-01-08,AAPL,24.781897,0.038422
5,2015-01-09,AAPL,24.808466,0.001072


**Create Dimension Table**

In [7]:
dim_asset = pd.DataFrame({
    "asset_id": range(1, 13),
    "ticker": [
        "AAPL", "MSFT", "NVDA", "JPM", "XOM",
        "GLD", "TLT", "SPY",
        "AMZN", "GOOGL", "META", "TSLA"
    ],
    "asset_name": [
        "Apple Inc.",
        "Microsoft Corp.",
        "Nvidia Corp.",
        "JPMorgan Chase & Co.",
        "Exxon Mobil Corp.",
        "SPDR Gold Shares",
        "iShares 20+ Year Treasury Bond ETF",
        "SPDR S&P 500 ETF Trust",
        "Amazon.com Inc.",
        "Alphabet Inc.",
        "Meta Platforms Inc.",
        "Tesla Inc."
    ],
    "asset_class": [
        "Equity", "Equity", "Equity", "Equity", "Equity",
        "Commodity", "Bond", "Benchmark",
        "Equity", "Equity", "Equity", "Equity"
    ],
    "sector": [
        "Technology", "Technology", "Technology", "Financials", "Energy",
        "Gold", "Treasury", "Market Index",
        "Consumer Discretionary", "Communication Services",
        "Communication Services", "Consumer Discretionary"
    ]
})

dim_asset

,asset_id,ticker,asset_name,asset_class,sector
0,1,AAPL,Apple Inc.,Equity,Technology
1,2,MSFT,Microsoft Corp.,Equity,Technology
2,3,NVDA,Nvidia Corp.,Equity,Technology
3,4,JPM,JPMorgan Chase & Co.,Equity,Financials
4,5,XOM,Exxon Mobil Corp.,Equity,Energy
5,6,GLD,SPDR Gold Shares,Commodity,Gold
6,7,TLT,iShares 20+ Year Treasury Bond ETF,Bond,Treasury
7,8,SPY,SPDR S&P 500 ETF Trust,Benchmark,Market Index
8,9,AMZN,Amazon.com Inc.,Equity,Consumer Discretionary
9,10,GOOGL,Alphabet Inc.,Equity,Communication Services


**Portfolio Weights**

In [8]:
portfolio_weights = pd.DataFrame({
    "ticker": [
        "AAPL", "MSFT", "NVDA", "JPM", "XOM",
        "GLD", "TLT", "AMZN", "GOOGL", "META", "TSLA"
    ],
    "weight": [
        0.10, 0.10, 0.10, 0.10, 0.08,
        0.15, 0.12, 0.08, 0.07, 0.05, 0.05
    ]
})

portfolio_weights["weight"].sum()

np.float64(1.0000000000000002)

**Portfolio Returns**

In [9]:
portfolio_daily = fact_asset_returns.merge(
    portfolio_weights,
    on="ticker",
    how="inner"
)

portfolio_daily["weighted_return"] = (
    portfolio_daily["daily_return"] * portfolio_daily["weight"]
)

fact_portfolio_returns = (
    portfolio_daily
    .groupby("return_date", as_index=False)["weighted_return"]
    .sum()
)

fact_portfolio_returns = fact_portfolio_returns.rename(columns={
    "weighted_return": "portfolio_daily_return"
})

fact_portfolio_returns["cumulative_return"] = (
    (1 + fact_portfolio_returns["portfolio_daily_return"]).cumprod() - 1
)

fact_portfolio_returns.head()

,return_date,portfolio_daily_return,cumulative_return
0,2015-01-05,-0.012453,-0.012453
1,2015-01-06,-0.007581,-0.019940
2,2015-01-07,0.002819,-0.017177
3,2015-01-08,0.013937,-0.003479
4,2015-01-09,-0.002176,-0.005648


**Create monthly portfolio returns**

In [10]:
monthly_returns = fact_portfolio_returns.copy()

monthly_returns["month"] = pd.to_datetime(
    monthly_returns["return_date"]
).dt.to_period("M").astype(str)

monthly_returns = (
    monthly_returns
    .groupby("month", as_index=False)["portfolio_daily_return"]
    .apply(lambda x: (1 + x).prod() - 1)
)

monthly_returns = monthly_returns.rename(columns={
    "portfolio_daily_return": "monthly_return"
})

monthly_returns.head()

,month,monthly_return
0,2015-01,0.000678
1,2015-02,0.042574
2,2015-03,-0.025170
3,2015-04,0.046533
4,2015-05,0.009634


**Export CSV files**

In [11]:
dim_asset.to_csv("data/processed/dim_asset.csv", index=False)
fact_asset_prices.to_csv("data/processed/fact_asset_prices.csv", index=False)
fact_asset_returns.to_csv("data/processed/fact_asset_returns.csv", index=False)
portfolio_weights.to_csv("data/processed/portfolio_weights.csv", index=False)
fact_portfolio_returns.to_csv("data/processed/fact_portfolio_returns.csv", index=False)
monthly_returns.to_csv("data/processed/monthly_portfolio_returns.csv", index=False)

**Create Database**

In [12]:
engine = create_engine("sqlite:///portfolio_analytics.db")

dim_asset.to_sql("dim_asset", engine, if_exists="replace", index=False)
fact_asset_prices.to_sql("fact_asset_prices", engine, if_exists="replace", index=False)
fact_asset_returns.to_sql("fact_asset_returns", engine, if_exists="replace", index=False)
portfolio_weights.to_sql("portfolio_weights", engine, if_exists="replace", index=False)
fact_portfolio_returns.to_sql("fact_portfolio_returns", engine, if_exists="replace", index=False)
monthly_returns.to_sql("monthly_portfolio_returns", engine, if_exists="replace", index=False)

136

**Confirm File Exists**

In [13]:
os.listdir()

['.config', 'portfolio_analytics.db', 'data', 'sample_data']

**Verifying the Data**

In [14]:
print("Prices:", fact_asset_prices.shape)
print("Returns:", fact_asset_returns.shape)
print("Portfolio:", fact_portfolio_returns.shape)
print("Monthly:", monthly_returns.shape)
print("Weights sum:", portfolio_weights["weight"].sum())

Prices: (34140, 3)
Returns: (34128, 4)
Portfolio: (2844, 3)
Monthly: (136, 2)
Weights sum: 1.0000000000000002


**Test SQL inside Colab**

In [15]:
pd.read_sql("""
SELECT *
FROM dim_asset;
""", engine)

,asset_id,ticker,asset_name,asset_class,sector
0,1,AAPL,Apple Inc.,Equity,Technology
1,2,MSFT,Microsoft Corp.,Equity,Technology
2,3,NVDA,Nvidia Corp.,Equity,Technology
3,4,JPM,JPMorgan Chase & Co.,Equity,Financials
4,5,XOM,Exxon Mobil Corp.,Equity,Energy
5,6,GLD,SPDR Gold Shares,Commodity,Gold
6,7,TLT,iShares 20+ Year Treasury Bond ETF,Bond,Treasury
7,8,SPY,SPDR S&P 500 ETF Trust,Benchmark,Market Index
8,9,AMZN,Amazon.com Inc.,Equity,Consumer Discretionary
9,10,GOOGL,Alphabet Inc.,Equity,Communication Services


In [16]:
pd.read_sql("""
SELECT
    r.return_date,
    r.ticker,
    a.asset_name,
    a.asset_class,
    a.sector,
    r.close_price,
    r.daily_return
FROM fact_asset_returns r
JOIN dim_asset a
    ON r.ticker = a.ticker
ORDER BY r.return_date, r.ticker
LIMIT 20;
""", engine)

,return_date,ticker,asset_name,asset_class,sector,close_price,daily_return
0,2015-01-05 00:00:00.000000,AAPL,Apple Inc.,Equity,Technology,23.532717,-0.028172
1,2015-01-05 00:00:00.000000,AMZN,Amazon.com Inc.,Equity,Consumer Discretionary,15.109500,-0.020517
2,2015-01-05 00:00:00.000000,GLD,SPDR Gold Shares,Commodity,Gold,115.800003,0.015077
3,2015-01-05 00:00:00.000000,GOOGL,Alphabet Inc.,Equity,Communication Services,25.760096,-0.019054
4,2015-01-05 00:00:00.000000,JPM,JPMorgan Chase & Co.,Equity,Financials,44.837734,-0.031045
5,2015-01-05 00:00:00.000000,META,Meta Platforms Inc.,Equity,Communication Services,76.588966,-0.016061
6,2015-01-05 00:00:00.000000,MSFT,Microsoft Corp.,Equity,Technology,39.401997,-0.009196
7,2015-01-05 00:00:00.000000,NVDA,Nvidia Corp.,Equity,Technology,0.474828,-0.016890
8,2015-01-05 00:00:00.000000,SPY,SPDR S&P 500 ETF Trust,Benchmark,Market Index,167.052612,-0.018060
9,2015-01-05 00:00:00.000000,TLT,iShares 20+ Year Treasury Bond ETF,Bond,Treasury,95.184158,0.015709


**Download the file**

In [52]:
files.download("portfolio_analytics.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:


files.download("data/processed/dim_asset.csv")
files.download("data/processed/fact_asset_returns.csv")
files.download("data/processed/fact_portfolio_returns.csv")
files.download("data/processed/monthly_portfolio_returns.csv")
files.download("data/processed/portfolio_weights.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>